Hello world of LLMs

In [40]:
import os
import openai


# Verify Jupyter sees your terminal environment token
github_token = os.environ.get("GITHUB_LLM_TOKEN")

client = openai.OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=github_token
)

response = client.chat.completions.create(
    model="Llama-3.3-70B-Instruct",
    messages=[
        {"role": "user", "content": "Hello Llama! I am setting up my LLM Zoomcamp environment."}
    ]
)

print(response.choices[0].message.content)

Setting up your environment for LLM Zoomcamp can be an exciting step. How can I assist you with the setup process? Are you encountering any issues or do you have questions about the requirements or installation steps? I'm here to help.


Calling LLM using prompt

In [41]:
def llm(prompt):
    response = client.chat.completions.create(
        model='Llama-3.3-70B-Instruct',
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # Extract the string content from the structured response
    return response.choices[0].message.content

In [42]:
llm("Hey, what's up?")

"Not much! Just here and ready to chat. What's on your mind? Want to talk about something in particular or just shoot the breeze?"

Asking questions that LLM isnt trained with (info in external DB)

In [43]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)


I'm happy you're interested in the course. However, I need a bit more information from you. Could you please tell me which course you're referring to and when it started? That way, I can provide you with more accurate guidance on whether you can still join or not.


Dataset - FAQ json data

In [44]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


In [45]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [46]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

Indexing with minsearch

In [47]:
from minsearch import Index

This terminology (text fields, keyword fields) comes from Elasticsearch, which in turn comes from Apache Lucene. 


In [48]:
index = Index(
    text_fields = ['question', 'section', 'answer'],
    keyword_fields = ['course'] )

In [49]:
index.fit(documents)

Trying a search

In [50]:
question = 'I just discovered the course. Can I join now?'

search_results = index.search(
    question,
    boost_dict={'question': 0.5, 'section': 2}, #more weightage given to question field than section field
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [51]:
#All questions
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Homework: Why does the content keep changing?',
 'I missed the first homework - can I still get a certificate?']

Boosting fields: All fields have a default boost of 1. Giving question a boost of 2 means it counts two times as much. Giving section 0.5 means it counts half as much. This is the same boosting mechanism used by Elasticsearch and Lucene.

In [52]:
question = 'Do I get a certificate for the course'

In [53]:
results = index.search(
    question,
    num_results=5,
    boost_dict={'question': 2, 'section': 0.5},
    filter_dict={'course': 'mlops-zoomcamp'}
)

In [54]:
[doc['question'] for doc in results]

['Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?',
 'Is completion of Homework necessary for a certificate?',
 'Criteria for getting a certificate?',
 'Course - Can I still join the course after the start date?']

Wrapping into a function

In [55]:
def search(question, course = 'llm-zoomcamp'):
    boost_dict = {'question' : 2.0, 'section' : 0.5}
    filter_dict = {'course' : course}

    return index.search(
    question, boost_dict = boost_dict, filter_dict = filter_dict, num_results = 5)

In [56]:
[doc['question'] for doc in results]

['Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?',
 'Is completion of Homework necessary for a certificate?',
 'Criteria for getting a certificate?',
 'Course - Can I still join the course after the start date?']

Building the prompt --> instructions + user's question + search results


Prompt bridge between Search and LLM

Bulding the context

In [57]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [58]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [59]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt =  USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [60]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
Do I get a certificate for the course

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You c

LLM - Sending the prompt to the LLM

In [61]:
response = client.chat.completions.create(
        model='Llama-3.3-70B-Instruct',
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

In [62]:
text_output = response.choices[0].message.content

In [63]:
print(text_output)

To get a certificate for the course, you need to:

1. Finish the course with a "live" cohort (not in self-paced mode).
2. Submit your project while submissions are still being accepted.
3. Pass the Capstone project.

Note that homework is not mandatory, but it's recommended for reinforcing concepts. Missing the first homework will not prevent you from getting a certificate, as long as you pass the Capstone project.


Message history 

In [64]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [70]:
def llm(developer_instructions,user_prompt, model):
    message_history = [
    {'role': 'system', 'content': developer_instructions},
    {'role': 'user', 'content': user_prompt}
    ]

    response = client.chat.completions.create(
        model=model,
        messages= message_history
    )

    print(response.choices[0].message.content)

In [71]:
llm(INSTRUCTIONS, prompt, 'Llama-3.3-70B-Instruct')

To get a certificate for the course, you need to:

1. Finish the course with a "live" cohort (not in self-paced mode).
2. Submit your project while submissions are still being accepted.
3. Pass the Capstone project.

Note that completing homework is not mandatory for getting a certificate, but it is recommended for reinforcing concepts. Peer-reviewing 3 capstone projects after submitting your own project is also a requirement for getting a certificate.


Full RAG = search + prompt + LLM

In [72]:
def rag(query, model = 'Llama-3.3-70B-Instruct'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, 'Llama-3.3-70B-Instruct')

In [73]:
answer = rag(query = 'I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
None


In [74]:
rag("Why is this course important")

I don't know. The provided context does not mention why the course is important. It only addresses specific questions related to course logistics, technical issues, and content.


In [76]:
rag("By whom is this toutorial taught?")

I don't know. The context does not mention who teaches the tutorial.
